# NYC Housing Data Profiling

This notebook performs exploratory data profiling on January 2026 samples
of NYC 311 housing complaints and HPD Housing Maintenance Code Violations.

The analysis examines data structure, missing values, categorical
distributions, date fields, duplicate records, and BBL-based geographic
overlap to assess data quality and inform the design of an integrated
housing data warehouse.

## 1. Load the two files

In [2]:
import pandas as pd

# Load 311 data
df_311 = pd.read_csv("nyc_311_january_2026.csv")

# Load HPD violations
df_hpd = pd.read_csv("hpd_violations_january_2026.csv")

print("311 shape:", df_311.shape)
print("HPD shape:", df_hpd.shape)

311 shape: (99584, 30)
HPD shape: (63228, 41)


## 2. Profile the columns

In [3]:
print("311 COLUMNS")
print(df_311.columns.tolist())

print("\nHPD COLUMNS")
print(df_hpd.columns.tolist())

311 COLUMNS
['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'descriptor_2', 'location_type', 'incident_zip', 'incident_address', 'street_name', 'address_type', 'city', 'status', 'resolution_description', 'resolution_action_updated_date', 'community_board', 'council_district', 'police_precinct', 'bbl', 'borough', 'x_coordinate_state_plane', 'y_coordinate_state_plane', 'open_data_channel_type', 'park_facility_name', 'park_borough', 'latitude', 'longitude', 'location']

HPD COLUMNS
['violationid', 'buildingid', 'registrationid', 'boroid', 'boro', 'housenumber', 'lowhousenumber', 'highhousenumber', 'streetname', 'streetcode', 'zip', 'apartment', 'story', 'block', 'lot', 'class', 'inspectiondate', 'approveddate', 'originalcertifybydate', 'originalcorrectbydate', 'certifieddate', 'ordernumber', 'novid', 'novdescription', 'novissueddate', 'currentstatusid', 'currentstatus', 'currentstatusdate', 'novtype', 'violationstatus', 'rentimpairing

In [4]:
print("311 DATA TYPES")
display(df_311.dtypes)

print("\nHPD DATA TYPES")
display(df_hpd.dtypes)

311 DATA TYPES


,0
unique_key,int64
created_date,object
closed_date,object
agency,object
agency_name,object
complaint_type,object
descriptor,object
descriptor_2,object
location_type,object
incident_zip,float64



HPD DATA TYPES


,0
violationid,int64
buildingid,int64
registrationid,int64
boroid,int64
boro,object
housenumber,object
lowhousenumber,object
highhousenumber,object
streetname,object
streetcode,int64


## 3. Look at missing values

In [5]:
print("311 MISSING VALUES")
missing_311 = pd.DataFrame({
    "missing_count": df_311.isna().sum(),
    "missing_percent": df_311.isna().mean() * 100
}).sort_values("missing_percent", ascending=False)

display(missing_311)

311 MISSING VALUES


,missing_count,missing_percent
descriptor_2,3360,3.374036
closed_date,2890,2.902073
bbl,296,0.297237
incident_zip,3,0.003013
council_district,3,0.003013
city,3,0.003013
location,3,0.003013
x_coordinate_state_plane,3,0.003013
longitude,3,0.003013
latitude,3,0.003013


In [6]:
print("HPD MISSING VALUES")
missing_hpd = pd.DataFrame({
    "missing_count": df_hpd.isna().sum(),
    "missing_percent": df_hpd.isna().mean() * 100
}).sort_values("missing_percent", ascending=False)

display(missing_hpd)

HPD MISSING VALUES


,missing_count,missing_percent
newcertifybydate,62730,99.212374
newcorrectbydate,62730,99.212374
certifieddate,32235,50.982160
apartment,15831,25.037958
novid,279,0.441260
originalcorrectbydate,279,0.441260
originalcertifybydate,279,0.441260
novissueddate,279,0.441260
novtype,279,0.441260
story,274,0.433352


## 4. Look at important categorical values

In [7]:
print("311 COMPLAINT TYPES")
display(df_311["complaint_type"].value_counts())

311 COMPLAINT TYPES


,count
complaint_type,
HEAT/HOT WATER,79928
UNSANITARY CONDITION,10961
PLUMBING,8695


In [8]:
print("311 BOROUGHS")
display(df_311["borough"].value_counts())

311 BOROUGHS


,count
borough,
BRONX,33676
BROOKLYN,27827
MANHATTAN,23321
QUEENS,13460
STATEN ISLAND,1298
Unspecified,2


In [9]:
print("311 STATUS")
display(df_311["status"].value_counts())

311 STATUS


,count
status,
Closed,96694
Open,2890


In [10]:
print("311 DESCRIPTORS")
display(df_311["descriptor"].value_counts().head(20))

311 DESCRIPTORS


,count
descriptor,
ENTIRE BUILDING,51163
APARTMENT ONLY,28765
PESTS,5656
MOLD,3360
WATER SUPPLY,2798
BASIN/SINK,1820
GARBAGE/RECYCLING STORAGE,1625
BATHTUB/SHOWER,1611
TOILET,1160


In [11]:
print("HPD BOROUGHS")
display(df_hpd["boro"].value_counts())

HPD BOROUGHS


,count
boro,
BRONX,22931
BROOKLYN,20032
MANHATTAN,10683
QUEENS,8148
STATEN ISLAND,1434


In [12]:
print("HPD VIOLATION CLASS")
display(df_hpd["class"].value_counts())

HPD VIOLATION CLASS


,count
class,
C,27212
B,26803
A,8949
I,264


In [13]:
print("HPD CURRENT STATUS")
display(df_hpd["currentstatus"].value_counts().head(20))

HPD CURRENT STATUS


,count
currentstatus,
VIOLATION DISMISSED,20922
VIOLATION CLOSED,18494
NOV SENT OUT,15536
NOT COMPLIED WITH,2116
FIRST NO ACCESS TO RE- INSPECT VIOLATION,1870
NOTICE OF ISSUANCE SENT TO TENANT,1400
INVALID CERTIFICATION,755
NOV CERTIFIED LATE,656
DEFECT LETTER ISSUED,540


In [14]:
print("HPD VIOLATION STATUS")
display(df_hpd["violationstatus"].value_counts())

HPD VIOLATION STATUS


,count
violationstatus,
Close,39502
Open,23726


In [15]:
print("HPD NOV TYPE")
display(df_hpd["novtype"].value_counts())

HPD NOV TYPE


,count
novtype,
Original,62949


## 5. Check geographic fields

In [16]:
print("311 BBL missing:")
print(df_311["bbl"].isna().sum())

print("HPD BBL missing:")
print(df_hpd["bbl"].isna().sum())

311 BBL missing:
296
HPD BBL missing:
24


In [17]:
print("311 BBL examples:")
display(df_311[["incident_address", "borough", "bbl"]].head(20))

311 BBL examples:


,incident_address,borough,bbl
0,139 HARMAN STREET,BROOKLYN,3.032760e+09
1,557 KNICKERBOCKER AVENUE,BROOKLYN,3.033350e+09
2,147 FRANKLIN STREET,BROOKLYN,3.025390e+09
3,2675 WEST 36 STREET,BROOKLYN,3.069620e+09
4,41-38 67 STREET,QUEENS,4.013060e+09
5,458 WYONA STREET,BROOKLYN,3.037910e+09
6,10563 FLATLANDS 4 STREET,BROOKLYN,3.082520e+09
7,189-22 116 ROAD,QUEENS,4.110590e+09
8,402 EAST 10 STREET,MANHATTAN,1.003790e+09
9,534 EAST 88 STREET,MANHATTAN,1.015840e+09


In [18]:
print("HPD BBL examples:")
display(df_hpd[["housenumber", "streetname", "boro", "block", "lot", "bbl"]].head(20))

HPD BBL examples:


,housenumber,streetname,boro,block,lot,bbl
0,390,PARKSIDE AVENUE,BROOKLYN,5055,46,3.050550e+09
1,143,STERLING STREET,BROOKLYN,1314,80,3.013140e+09
2,143,STERLING STREET,BROOKLYN,1314,80,3.013140e+09
3,2901,GRAND CONCOURSE,BRONX,3319,101,2.033190e+09
4,2430,MARION AVENUE,BRONX,3025,1,2.030250e+09
5,106-44,GUY R BREWER BOULEVARD,QUEENS,10129,29,4.101290e+09
6,649,EMPIRE BOULEVARD,BROOKLYN,1312,55,3.013120e+09
7,228,WEST 116 STREET,MANHATTAN,1831,47,1.018310e+09
8,26,GATES AVENUE,BROOKLYN,1976,7,3.019760e+09
9,26,GATES AVENUE,BROOKLYN,1976,7,3.019760e+09


## 6. Check BBL reliability

In [19]:
print("311 BBL unique values:", df_311["bbl"].nunique())
print("311 total rows:", len(df_311))

print("HPD BBL unique values:", df_hpd["bbl"].nunique())
print("HPD total rows:", len(df_hpd))

311 BBL unique values: 18428
311 total rows: 99584
HPD BBL unique values: 8503
HPD total rows: 63228


In [20]:
print("311 missing BBL %:",
      df_311["bbl"].isna().mean() * 100)

print("HPD missing BBL %:",
      df_hpd["bbl"].isna().mean() * 100)

311 missing BBL %: 0.2972365038560411
HPD missing BBL %: 0.03795786676788764


In [21]:
print(df_311["bbl"].dtype)
print(df_hpd["bbl"].dtype)

float64
float64


## 7. Check dates

In [22]:
df_311["created_date"] = pd.to_datetime(df_311["created_date"])
df_311["closed_date"] = pd.to_datetime(df_311["closed_date"])

print(df_311["created_date"].min())
print(df_311["created_date"].max())

2026-01-01 00:00:25
2026-01-31 23:59:54


In [23]:
df_hpd["inspectiondate"] = pd.to_datetime(df_hpd["inspectiondate"])

print(df_hpd["inspectiondate"].min())
print(df_hpd["inspectiondate"].max())

2026-01-01 00:00:00
2026-01-31 00:00:00


## 8. Initial ID Uniqueness Check

In [24]:
print("311 total rows:", len(df_311))
print("311 unique complaint IDs:", df_311["unique_key"].nunique())

311 total rows: 99584
311 unique complaint IDs: 76782


In [25]:
print("HPD total rows:", len(df_hpd))
print("HPD unique violation IDs:", df_hpd["violationid"].nunique())

HPD total rows: 63228
HPD unique violation IDs: 58810


## 9. Look at 311 complaint types

In [26]:
complaint_counts = (
    df_311["complaint_type"]
    .value_counts()
    .rename_axis("complaint_type")
    .reset_index(name="complaint_count")
)

display(complaint_counts)

,complaint_type,complaint_count
0,HEAT/HOT WATER,79928
1,UNSANITARY CONDITION,10961
2,PLUMBING,8695


In [27]:
complaint_counts["percentage"] = (
    complaint_counts["complaint_count"]
    / complaint_counts["complaint_count"].sum()
    * 100
)

display(complaint_counts)

,complaint_type,complaint_count,percentage
0,HEAT/HOT WATER,79928,80.261889
1,UNSANITARY CONDITION,10961,11.006788
2,PLUMBING,8695,8.731322


## 10. Look at HPD violation classes

In [28]:
# Count HPD violations by class
hpd_class_counts = df_hpd["class"].value_counts()

# Calculate percentages
hpd_class_percentages = (hpd_class_counts / len(df_hpd) * 100).round(2)

# Combine counts and percentages into one table
hpd_class_summary = pd.DataFrame({
    "Count": hpd_class_counts,
    "Percentage": hpd_class_percentages
})

display(hpd_class_summary)

,Count,Percentage
class,,
C,27212,43.04
B,26803,42.39
A,8949,14.15
I,264,0.42


## 11. Geographic Comparison: BBL Overlap Between Datasets

In [29]:
# Convert BBL to string in both datasets
df_311["bbl"] = df_311["bbl"].astype("string")
df_hpd["bbl"] = df_hpd["bbl"].astype("string")

# Remove missing BBLs for this comparison
bbl_311 = set(df_311["bbl"].dropna())
bbl_hpd = set(df_hpd["bbl"].dropna())

matching_bbls = bbl_311.intersection(bbl_hpd)

print("Unique 311 BBLs:", len(bbl_311))
print("Unique HPD BBLs:", len(bbl_hpd))
print("Matching BBLs:", len(matching_bbls))

Unique 311 BBLs: 18428
Unique HPD BBLs: 8503
Matching BBLs: 6957


In [30]:
print(
    "Percent of 311 BBLs also found in HPD:",
    len(matching_bbls) / len(bbl_311) * 100
)

Percent of 311 BBLs also found in HPD: 37.752333405687


In [31]:
# BBL analysis: how many 311 records have a BBL that also appears in HPD?

# Get the set of BBLs that appear in HPD
hpd_bbls = set(df_hpd["bbl"].dropna())

# Check whether each 311 record's BBL appears in HPD
df_311["bbl_matches_hpd"] = df_311["bbl"].isin(hpd_bbls)

# Count matching and non-matching records
display(df_311["bbl_matches_hpd"].value_counts())

# Calculate percentages
bbl_match_percent = df_311["bbl_matches_hpd"].mean() * 100

print(
    f"Percentage of 311 records with a BBL found in HPD: "
    f"{bbl_match_percent:.2f}%"
)

,count
bbl_matches_hpd,
True,62524
False,37060


Percentage of 311 records with a BBL found in HPD: 62.79%


In [32]:
# BBL matching percentage by borough

bbl_match_by_borough = (
    df_311.groupby("borough")["bbl_matches_hpd"]
    .mean()
    .mul(100)
    .round(2)
    .reset_index(name="match_percent")
)

display(bbl_match_by_borough)

,borough,match_percent
0,BRONX,70.21
1,BROOKLYN,60.99
2,MANHATTAN,55.53
3,QUEENS,60.78
4,STATEN ISLAND,59.86
5,Unspecified,0.00


In [33]:
bbl_summary = pd.DataFrame({
    "Measure": [
        "Shared Unique BBLs",
        "311 Records with HPD-Matching BBL",
        "311 Records Total",
        "311 Records with HPD-Matching BBL (%)"
    ],
    "Value": [
        len(matching_bbls),
        df_311["bbl_matches_hpd"].sum(),
        len(df_311),
        f"{bbl_match_percent:.2f}%"
    ]
})

display(bbl_summary)

,Measure,Value
0,Shared Unique BBLs,6957
1,311 Records with HPD-Matching BBL,62524
2,311 Records Total,99584
3,311 Records with HPD-Matching BBL (%),62.79%


## 12. Small sample of each dataset

In [34]:
display(df_311.head(5))

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,...,borough,x_coordinate_state_plane,y_coordinate_state_plane,open_data_channel_type,park_facility_name,park_borough,latitude,longitude,location,bbl_matches_hpd
0,67634303,2026-01-27 04:55:36,2026-02-02 13:06:50,HPD,Department of Housing Preservation and Develop...,HEAT/HOT WATER,ENTIRE BUILDING,NO HEAT,RESIDENTIAL BUILDING,11221.0,...,BROOKLYN,1005894.0,193145.0,ONLINE,Unspecified,BROOKLYN,40.696789,-73.921946,"{'type': 'Point', 'coordinates': [-73.92194570...",False
1,67641635,2026-01-27 19:38:59,2026-02-02 14:12:31,HPD,Department of Housing Preservation and Develop...,HEAT/HOT WATER,ENTIRE BUILDING,NO HEAT,RESIDENTIAL BUILDING,11221.0,...,BROOKLYN,1007788.0,193216.0,ONLINE,Unspecified,BROOKLYN,40.696979,-73.915115,"{'type': 'Point', 'coordinates': [-73.91511516...",False
2,67651491,2026-01-27 04:48:44,2026-02-02 11:36:53,HPD,Department of Housing Preservation and Develop...,HEAT/HOT WATER,ENTIRE BUILDING,NO HEAT,RESIDENTIAL BUILDING,11222.0,...,BROOKLYN,995927.0,205775.0,ONLINE,Unspecified,BROOKLYN,40.731474,-73.957868,"{'type': 'Point', 'coordinates': [-73.95786756...",False
3,67662871,2026-01-28 09:54:09,2026-02-03 11:51:51,HPD,Department of Housing Preservation and Develop...,HEAT/HOT WATER,ENTIRE BUILDING,NO HEAT AND NO HOT WATER,RESIDENTIAL BUILDING,11224.0,...,BROOKLYN,983498.0,150561.0,ONLINE,Unspecified,BROOKLYN,40.579931,-74.002707,"{'type': 'Point', 'coordinates': [-74.00270717...",True
4,67664882,2026-01-28 00:13:43,2026-02-03 21:11:39,HPD,Department of Housing Preservation and Develop...,HEAT/HOT WATER,ENTIRE BUILDING,NO HEAT,RESIDENTIAL BUILDING,11377.0,...,QUEENS,1012601.0,210320.0,PHONE,Unspecified,QUEENS,40.743911,-73.897686,"{'type': 'Point', 'coordinates': [-73.89768606...",True


In [35]:
display(df_hpd.head(5))

,violationid,buildingid,registrationid,boroid,boro,housenumber,lowhousenumber,highhousenumber,streetname,streetcode,...,latitude,longitude,communityboard,councildistrict,censustract,bin,bbl,nta,newcertifybydate,newcorrectbydate
0,18544677,352250,309008,3,BROOKLYN,390,390,390,PARKSIDE AVENUE,69530,...,40.655754,-73.957987,9.0,40.0,79601.0,3115969.0,3050550046.0,Prospect Lefferts Gardens-Wingate,NaN,NaN
1,18544702,377021,328742,3,BROOKLYN,143,143,143,STERLING STREET,81530,...,40.662816,-73.956015,9.0,40.0,327.0,3034788.0,3013140080.0,Prospect Lefferts Gardens-Wingate,NaN,NaN
2,18544704,377021,328742,3,BROOKLYN,143,143,143,STERLING STREET,81530,...,40.662816,-73.956015,9.0,40.0,327.0,3034788.0,3013140080.0,Prospect Lefferts Gardens-Wingate,NaN,NaN
3,18544796,81897,211064,2,BRONX,2901,2901,2901,GRAND CONCOURSE,36420,...,40.871113,-73.889359,7.0,14.0,40701.0,2017601.0,2033190101.0,Bedford Park,NaN,NaN
4,18544969,94333,206567,2,BRONX,2430,2426,2430,MARION AVENUE,49420,...,40.859826,-73.894377,5.0,15.0,39902.0,2011056.0,2030250001.0,Fordham Heights,NaN,NaN


In [36]:
print("Sample 311 records")
display(df_311[
    ["unique_key", "created_date", "complaint_type",
     "descriptor", "status", "borough", "bbl"]
].head())

Sample 311 records


,unique_key,created_date,complaint_type,descriptor,status,borough,bbl
0,67634303,2026-01-27 04:55:36,HEAT/HOT WATER,ENTIRE BUILDING,Closed,BROOKLYN,3032760001.0
1,67641635,2026-01-27 19:38:59,HEAT/HOT WATER,ENTIRE BUILDING,Closed,BROOKLYN,3033350001.0
2,67651491,2026-01-27 04:48:44,HEAT/HOT WATER,ENTIRE BUILDING,Closed,BROOKLYN,3025390021.0
3,67662871,2026-01-28 09:54:09,HEAT/HOT WATER,ENTIRE BUILDING,Closed,BROOKLYN,3069620011.0
4,67664882,2026-01-28 00:13:43,HEAT/HOT WATER,ENTIRE BUILDING,Closed,QUEENS,4013060130.0


In [37]:
print("Sample HPD records")
display(df_hpd[
    ["violationid", "buildingid", "inspectiondate",
     "class", "currentstatus", "violationstatus", "boro", "bbl"]
].head())

Sample HPD records


,violationid,buildingid,inspectiondate,class,currentstatus,violationstatus,boro,bbl
0,18544677,352250,2026-01-01,C,VIOLATION DISMISSED,Close,BROOKLYN,3050550046.0
1,18544702,377021,2026-01-01,B,NOV SENT OUT,Open,BROOKLYN,3013140080.0
2,18544704,377021,2026-01-01,B,NOV SENT OUT,Open,BROOKLYN,3013140080.0
3,18544796,81897,2026-01-01,C,VIOLATION DISMISSED,Close,BRONX,2033190101.0
4,18544969,94333,2026-01-01,C,VIOLATION DISMISSED,Close,BRONX,2030250001.0


## 13. How many IDs are repeated?

In [38]:
# Find repeated 311 complaint IDs

repeated_311_ids = (
    df_311["unique_key"]
    .value_counts()
)

print("Number of unique complaint IDs:", df_311["unique_key"].nunique())

print(
    "Number of complaint IDs appearing more than once:",
    (repeated_311_ids > 1).sum()
)

print(
    "Maximum number of rows for one complaint ID:",
    repeated_311_ids.max()
)

Number of unique complaint IDs: 76782
Number of complaint IDs appearing more than once: 22802
Maximum number of rows for one complaint ID: 2


In [39]:
# Show examples of repeated 311 IDs

duplicate_311_ids = repeated_311_ids[repeated_311_ids > 1].head(10).index

display(
    df_311[
        df_311["unique_key"].isin(duplicate_311_ids)
    ][
        [
            "unique_key",
            "created_date",
            "closed_date",
            "complaint_type",
            "descriptor",
            "status",
            "resolution_description",
            "bbl"
        ]
    ].sort_values("unique_key")
)

,unique_key,created_date,closed_date,complaint_type,descriptor,status,resolution_description,bbl
26669,67596036,2026-01-24 12:14:30,2026-01-26 20:31:24,HEAT/HOT WATER,APARTMENT ONLY,Closed,HPD responded to a complaint of no heat or hot...,2026630007.0
53331,67596036,2026-01-24 12:14:30,2026-01-26 20:31:24,HEAT/HOT WATER,APARTMENT ONLY,Closed,HPD responded to a complaint of no heat or hot...,2026630007.0
53392,67596098,2026-01-24 16:38:53,2026-01-27 18:43:57,HEAT/HOT WATER,APARTMENT ONLY,Closed,HPD inspected this condition so the complaint ...,2048350054.0
26673,67596098,2026-01-24 16:38:53,2026-01-27 18:43:57,HEAT/HOT WATER,APARTMENT ONLY,Closed,HPD inspected this condition so the complaint ...,2048350054.0
26674,67596099,2026-01-24 14:01:47,2026-01-26 14:17:13,HEAT/HOT WATER,APARTMENT ONLY,Closed,The Department of Housing Preservation and Dev...,2030690078.0
53393,67596099,2026-01-24 14:01:47,2026-01-26 14:17:13,HEAT/HOT WATER,APARTMENT ONLY,Closed,The Department of Housing Preservation and Dev...,2030690078.0
53398,67596104,2026-01-24 15:06:21,2026-01-29 12:21:10,HEAT/HOT WATER,APARTMENT ONLY,Closed,HPD called the telephone number on file for th...,3012030049.0
26676,67596104,2026-01-24 15:06:21,2026-01-29 12:21:10,HEAT/HOT WATER,APARTMENT ONLY,Closed,HPD called the telephone number on file for th...,3012030049.0
53473,67596182,2026-01-24 22:22:51,2026-01-26 09:04:45,HEAT/HOT WATER,ENTIRE BUILDING,Closed,HPD called the telephone number on file for th...,2048200042.0
26679,67596182,2026-01-24 22:22:51,2026-01-26 09:04:45,HEAT/HOT WATER,ENTIRE BUILDING,Closed,HPD called the telephone number on file for th...,2048200042.0


In [40]:
# Find repeated HPD violation IDs

repeated_hpd_ids = (
    df_hpd["violationid"]
    .value_counts()
)

print("Number of unique violation IDs:", df_hpd["violationid"].nunique())

print(
    "Number of violation IDs appearing more than once:",
    (repeated_hpd_ids > 1).sum()
)

print(
    "Maximum number of rows for one violation ID:",
    repeated_hpd_ids.max()
)

Number of unique violation IDs: 58810
Number of violation IDs appearing more than once: 4418
Maximum number of rows for one violation ID: 2


In [41]:
# Show examples of repeated HPD violation IDs

duplicate_hpd_ids = repeated_hpd_ids[repeated_hpd_ids > 1].head(10).index

display(
    df_hpd[
        df_hpd["violationid"].isin(duplicate_hpd_ids)
    ][
        [
            "violationid",
            "buildingid",
            "inspectiondate",
            "class",
            "currentstatus",
            "violationstatus",
            "bbl"
        ]
    ].sort_values("violationid")
)

,violationid,buildingid,inspectiondate,class,currentstatus,violationstatus,bbl
3299,18546923,488184,2026-01-03,B,VIOLATION DISMISSED,Close,4097360108.0
4030,18546923,488184,2026-01-03,B,VIOLATION DISMISSED,Close,4097360108.0
4031,18546925,488184,2026-01-03,B,VIOLATION DISMISSED,Close,4097360108.0
3301,18546925,488184,2026-01-03,B,VIOLATION DISMISSED,Close,4097360108.0
4032,18546926,488184,2026-01-03,B,VIOLATION DISMISSED,Close,4097360108.0
3302,18546926,488184,2026-01-03,B,VIOLATION DISMISSED,Close,4097360108.0
3305,18547620,66476,2026-01-03,B,VIOLATION DISMISSED,Close,2031450019.0
4034,18547620,66476,2026-01-03,B,VIOLATION DISMISSED,Close,2031450019.0
23277,18567348,212884,2026-01-13,C,VIOLATION DISMISSED,Close,3072630124.0
24586,18567348,212884,2026-01-13,C,VIOLATION DISMISSED,Close,3072630124.0


In [42]:
# How many rows are completely identical?

print(
    "Completely duplicated 311 rows:",
    df_311.duplicated().sum()
)

Completely duplicated 311 rows: 22802


In [43]:
print(
    "Completely duplicated HPD rows:",
    df_hpd.duplicated().sum()
)

Completely duplicated HPD rows: 4418


In [44]:
duplicate_summary = pd.DataFrame({
    "Dataset": [
        "311 Service Requests",
        "HPD Housing Maintenance Code Violations"
    ],
    "Total Rows": [
        len(df_311),
        len(df_hpd)
    ],
    "Exact Duplicate Rows": [
        df_311.duplicated().sum(),
        df_hpd.duplicated().sum()
    ]
})

duplicate_summary["Duplicate Percentage"] = (
    duplicate_summary["Exact Duplicate Rows"]
    / duplicate_summary["Total Rows"] * 100
).round(2)

display(duplicate_summary)

,Dataset,Total Rows,Exact Duplicate Rows,Duplicate Percentage
0,311 Service Requests,99584,22802,22.90
1,HPD Housing Maintenance Code Violations,63228,4418,6.99
